# 73 — Run Blind-A with config 180 (wRRF + fine-tuned BGE-M3, Submission 1) → submission zip

Produces `prediction.json` for CodaBench by running the **Submission 1** pipeline
over the 80 Blind-A queries:

```
wRRF(BM25 + dense_lyrics + fine-tuned BGE-M3) → ProRank → v5-kto Qwen-3B responder
```

**Config 180** points at `OrRim123/recsys2026-bge-m3-music-v1-merged` (the model
trained in nb 70). CMQR is OFF for a clean read on the fine-tune's standalone
contribution; re-enable in Submission 2/3 after Stage A signal is established.

Per spec §9 (`docs/superpowers/specs/2026-05-18-ndcg-stretch-design.md`),
Submission 1 gate is **composite ≥ 0.21** (no regression vs the v5-kto baseline).
Expected wallclock: ~35–65 min on Blackwell.

> SID is dropped (per memory `project_sid_closed_2026_05_18`). This config uses
> the post-SID three-stream wRRF (BM25 + dense_lyrics + BGE-M3-FT).

In [ ]:
# 1) CONFIG — single source of truth for this submission.

# --- Identity --------------------------------------------------
BRANCH         = 'fresh-model'
TID            = '180-wrrf-bge-m3-ft-v5kto-blindA'   # config file name in music-crs-baselines/config/<TID>.yaml
HUB_USER       = 'OrRim123'
RUN_NAME       = 'bge-m3-music-v1'                    # matches nb 70 — used only for preflight checks

# --- Derived paths (do not edit usually) -----------------------
HUB_REPO_MERGED   = f'{HUB_USER}/recsys2026-{RUN_NAME}-merged'
EMBED_LABEL       = f'{RUN_NAME}-merged'
SAFE_MODEL        = HUB_REPO_MERGED.replace('/', '_')
CATALOG_PKL       = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/dense_local/{SAFE_MODEL}/{EMBED_LABEL}/track_embeddings.pkl'
PRED_PATH         = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
INFERENCE_LOG     = f'/content/drive/MyDrive/recsys2026_retrieval_v2_cache/blindA_{TID}_log.txt'
SUBMISSION_DATE   = None   # auto-fills with today; override with 'YYYY-MM-DD' for replays
SUBMISSION_LABEL  = f'retrieval-v2-{TID}'

# --- Inference knobs ------------------------------------------
INFERENCE_BATCH_SIZE = 32

print('Config loaded:')
for k in ('BRANCH','TID','HUB_REPO_MERGED','CATALOG_PKL','PRED_PATH','INFERENCE_BATCH_SIZE'):
    print(f'  {k} = {globals()[k]!r}')


In [ ]:
# 2) Setup — clone branch + HF auth + Drive mount + symlinks + deps.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('dense', 'recsys2026_dense_cache'),
    # Fine-tuned BGE-M3 catalog pickle: written by nb 70 cell 6 to
    # MyDrive/recsys2026_retrieval_v2_cache/dense_local/. Without this
    # symlink, DENSE_LOCAL FileNotFoundError on inference.
    ('dense_local', 'recsys2026_retrieval_v2_cache/dense_local'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Inference deps. Mirrors notebook 41 cell 5's install set.
!pip install -q --upgrade \
    'peft>=0.10' 'transformers>=4.40' 'accelerate>=0.30' 'torchao>=0.17' \
    'bm25s>=0.3.0,<0.4' 'sentence-transformers' \
    'datasets' 'pandas<3.0' 'tqdm' 'omegaconf' 'pyyaml' \
    'trl>=0.12.0'


In [ ]:
# 3) Preflight + inference. Validates all required artifacts before
# spending ~45 min on Blind-A inference; clear error if anything's missing.

import os
from huggingface_hub import HfApi
from pathlib import Path

# Preflight 1: fine-tuned merged model exists on Hub.
try:
    HfApi().model_info(HUB_REPO_MERGED)
    print(f'[preflight] ✓ Hub model reachable: {HUB_REPO_MERGED}')
except Exception as e:
    raise RuntimeError(
        f'[preflight] Hub model NOT reachable: {HUB_REPO_MERGED!r}\n'
        f'  Likely cause: nb 70 cell 4 has not finished AND pushed the merged adapter yet.\n'
        f'  Original error: {e}'
    )

# Preflight 2: fine-tuned catalog pickle on Drive (written by nb 70 cell 6).
if not Path(CATALOG_PKL).exists():
    raise FileNotFoundError(
        f'[preflight] Catalog pickle MISSING: {CATALOG_PKL}\n'
        f'  Re-run cell 6 of nb 70 to encode the catalog with the fine-tuned model.'
    )
_sz_mb = Path(CATALOG_PKL).stat().st_size / (1024 * 1024)
print(f'[preflight] ✓ Catalog pickle exists ({_sz_mb:.1f} MB)')

# Preflight 3: config file exists in the music-crs-baselines/config/ dir.
_config_path = Path(f'music-crs-baselines/config/{TID}.yaml')
if not _config_path.exists():
    raise FileNotFoundError(
        f'[preflight] Config MISSING: {_config_path}. '
        f'Check TID spelling in CONFIG cell.'
    )
print(f'[preflight] ✓ Config file exists: {_config_path}')

# Preflight 4: dense_lyrics cache (used by the wRRF dense_lyrics stream).
_dense_lyrics_root = Path('/content/drive/MyDrive/recsys2026_dense_cache')
if _dense_lyrics_root.exists() and any(_dense_lyrics_root.iterdir()):
    print(f'[preflight] ✓ dense_lyrics cache present')
else:
    print(f'[preflight] ⚠ dense_lyrics cache appears empty at {_dense_lyrics_root}. '
          f'Inference may rebuild it (slower) or fail. Check earlier setup notebooks.')

print(f'\n[preflight] OK — kicking off Blind-A inference (~30-60 min on Blackwell)')
print(f'[preflight] live log streaming to {INFERENCE_LOG}')
print(f'[preflight] tail -f that file in a separate Colab tab to watch progress.\n')

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py \
    --tid {TID} \
    --batch_size {INFERENCE_BATCH_SIZE} \
    2>&1 | tee {INFERENCE_LOG} | tail -80


In [ ]:
# 4) Validate prediction.json (schema + record count + catalog-membership).
%cd /content/recsys2026
import json, subprocess, sys
from pathlib import Path
sys.path.insert(0, '/content/recsys2026')
from scripts.precheck_prediction import precheck, _load_catalog

preds = json.load(open(PRED_PATH))
n_entries = len(preds) if isinstance(preds, list) else len(preds.keys())
print(f'[validate] prediction file: {n_entries} entries')
assert n_entries == 80, f'EXPECTED 80, got {n_entries} — DO NOT submit'
sample_keys = list(preds[0].keys()) if isinstance(preds, list) else list(list(preds.values())[0].keys())
print(f'[validate] sample entry keys: {sample_keys}')

# Strict precheck: catalog membership + schema invariants.
catalog = _load_catalog('talkpl-ai/TalkPlayData-Challenge-Track-Metadata')
result = precheck(Path(PRED_PATH), catalog=catalog, expected_n=80)
assert result['ok'], (
    f'precheck FAILED with {len(result["errors"])} errors; '
    f'first 5: {result["errors"][:5]}'
)
print(f'[validate] ✓ precheck passed (n_records={result["n_records"]})')

# Schema validator (catches different bugs from precheck).
rc = subprocess.call(['python', 'scripts/validate_prediction.py', '--input', PRED_PATH, '--split', 'blindA'])
assert rc == 0, f'validate_prediction.py FAILED (rc={rc}) — do not submit'
print('[validate] ✓ schema validator passed')

# Peek at the first prediction to eyeball quality.
sample = preds[0] if isinstance(preds, list) else list(preds.values())[0]
print(f'\n[validate] first prediction sample:')
for k, v in sample.items():
    if isinstance(v, str) and len(v) > 200:
        print(f'  {k}: {v[:200]}...')
    elif isinstance(v, list) and len(v) > 5:
        print(f'  {k}: {v[:5]} ... ({len(v)} items)')
    else:
        print(f'  {k}: {v!r}')


In [ ]:
# 5) Zip for CodaBench. prediction.json MUST be at the ROOT of the zip
# (per memory project_codabench_submission: server reads /app/input/res/prediction.json).
import os, zipfile, datetime
_date = SUBMISSION_DATE or datetime.date.today().strftime('%Y-%m-%d')
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{_date}-{SUBMISSION_LABEL}.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')
print(f'[zip] ready: {zip_path}')
print(f'[zip] size: {os.path.getsize(zip_path) / 1024:.1f} KB')
print(f'\nDownload + upload to CodaBench at https://www.codabench.org/competitions/\n'
      f'After scoring, log the result via:\n'
      f'  !python scripts/blind_a_score_tracker.py append --tid {TID} \\\n'
      f'      --composite <X> --ndcg <Y> --llm <Z> --lex_div <W> --cat_div <V>')


## After the run

1. Download the zip from Drive: `/content/drive/MyDrive/recsys2026_submissions/<date>-retrieval-v2-{TID}.zip`
2. Upload to CodaBench (https://www.codabench.org/competitions/).
3. Append the composite + nDCG@20 + LLM + lex_div + cat_div scores to the score tracker:
   ```
   !python scripts/blind_a_score_tracker.py append --tid <TID> \
       --composite <X> --ndcg <Y> --llm <Z> --lex_div <W> --cat_div <V>
   ```
4. **If composite ≥ 0.21** (Submission 1 gate, spec §9): retrieval v2 Stage A is
   shippable. Proceed to Stage B (cross-encoder fine-tune, nb 71).
5. **If composite < 0.21**: check the axis breakdown — if nDCG dropped, the
   fine-tune regressed and we need to debug; if LLM/lex_div tanked, the
   responder context got polluted by the new top-N candidates (rare on a
   retrieval-only change). Consult
   `docs/superpowers/specs/2026-05-18-ndcg-stretch-design.md` §10 for abort rules.